In [1]:
import os
import sys
sys.path.append("src")

import pandas as pd
import seaborn as sns

from wildfire_gnn.features.feature_analysis import (
    compute_feature_summary,
    plot_feature_distributions,
    compute_feature_target_correlation,
    plot_feature_vs_target,
    compute_split_shift
)

In [5]:
os.chdir("..")
df = pd.read_csv("data/processed/baseline_dataset.csv")
df.head()

,CFL.img,FSP_Index.img,Ignition_Prob.img,Struct_Exp_Index.img,Fuel_Models.img,row_norm,col_norm,target,row_index,col_index
0,0.167812,0.072349,0.096481,0.072349,101.0,0.284360,0.284428,0.019862,540,537
1,0.167812,0.072349,0.096481,0.072349,141.0,0.103739,0.290784,0.023607,197,549
2,0.167812,0.072349,0.096481,0.072349,101.0,0.700369,0.319915,0.052330,1330,604
3,0.167812,0.072349,0.096481,0.072349,121.0,0.091101,0.598517,0.032072,173,1130
4,0.167812,0.072349,0.096481,0.072349,104.0,0.966825,0.660487,0.017760,1836,1247


In [6]:
print("Shape:", df.shape)
df.describe()

Shape: (300000, 10)


,CFL.img,FSP_Index.img,Ignition_Prob.img,Struct_Exp_Index.img,Fuel_Models.img,row_norm,col_norm,target,row_index,col_index
count,3.000000e+05,3.000000e+05,3.000000e+05,3.000000e+05,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000
mean,-3.512800e-09,1.945668e-11,-3.949098e-09,1.945668e-11,137.337023,0.410381,0.376464,0.025153,779.313027,710.763933
std,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,29.096457,0.245783,0.188600,0.033476,466.742473,356.077108
min,-1.118292e+01,-3.003882e+01,-1.830590e+01,-3.003882e+01,91.000000,0.001580,0.001589,0.000023,3.000000,3.000000
25%,1.678120e-01,7.234921e-02,9.648060e-02,7.234921e-02,121.000000,0.204318,0.238877,0.003818,388.000000,451.000000
50%,1.678120e-01,7.234921e-02,9.648060e-02,7.234921e-02,141.000000,0.386519,0.331568,0.012934,734.000000,626.000000
75%,1.678120e-01,7.234921e-02,9.648060e-02,7.234921e-02,161.000000,0.573460,0.484110,0.032171,1089.000000,914.000000
max,1.678120e-01,7.234921e-02,9.648060e-02,7.234921e-02,189.000000,0.998947,0.998411,0.249303,1897.000000,1885.000000


In [8]:
import numpy as np

splits = np.load("data/processed/baseline_splits_spatial.npz")

train_idx = splits["train_idx"]
val_idx = splits["val_idx"]
test_idx = splits["test_idx"]

df["split"] = "unknown"

df.loc[train_idx, "split"] = "train"
df.loc[val_idx, "split"] = "val"
df.loc[test_idx, "split"] = "test"

df["split"].value_counts()

split
train    199167
test      60115
val       40718
Name: count, dtype: int64

In [9]:
summary = compute_feature_summary(
    df.drop(columns=["split"]),
    save_path="reports/tables/feature_summary.csv"
)
summary

,count,mean,std,min,25%,50%,75%,max,median,skew,missing
CFL.img,300000.0,-3.512800e-09,1.000002,-11.182920,0.167812,0.167812,0.167812,0.167812,0.167812,-7.400692,0
FSP_Index.img,300000.0,1.945668e-11,1.000002,-30.038822,0.072349,0.072349,0.072349,0.072349,0.072349,-17.672006,0
Ignition_Prob.img,300000.0,-3.949098e-09,1.000002,-18.305900,0.096481,0.096481,0.096481,0.096481,0.096481,-12.458611,0
Struct_Exp_Index.img,300000.0,1.945668e-11,1.000002,-30.038822,0.072349,0.072349,0.072349,0.072349,0.072349,-17.672006,0
Fuel_Models.img,300000.0,1.373370e+02,29.096457,91.000000,121.000000,141.000000,161.000000,189.000000,141.000000,0.311264,0
row_norm,300000.0,4.103807e-01,0.245783,0.001580,0.204318,0.386519,0.573460,0.998947,0.386519,0.541367,0
col_norm,300000.0,3.764639e-01,0.188600,0.001589,0.238877,0.331568,0.484110,0.998411,0.331568,0.830714,0
target,300000.0,2.515261e-02,0.033476,0.000023,0.003818,0.012934,0.032171,0.249303,0.012934,2.531839,0
row_index,300000.0,7.793130e+02,466.742473,3.000000,388.000000,734.000000,1089.000000,1897.000000,734.000000,0.541367,0
col_index,300000.0,7.107639e+02,356.077108,3.000000,451.000000,626.000000,914.000000,1885.000000,626.000000,0.830714,0


In [14]:
feature_cols = [
    "CFL.img",
    "FSP_Index.img",
    "Ignition_Prob.img",
    "Struct_Exp_Index.img",
    "Fuel_Models.img",
    "row_norm",
    "col_norm",
    "target",
]

analysis_df = df[feature_cols].copy()

corr_df = compute_feature_target_correlation(
    analysis_df,
    target_col="target",
    save_path="reports/tables/feature_target_correlation.csv"
)

corr_df

,feature,pearson,spearman
5,row_norm,0.200904,0.292256
0,CFL.img,0.067459,0.126323
6,col_norm,-0.004796,0.117438
2,Ignition_Prob.img,0.029998,0.044031
1,FSP_Index.img,0.021462,0.033268
3,Struct_Exp_Index.img,0.021462,0.033268
4,Fuel_Models.img,-0.151608,-0.250589


In [12]:
plot_feature_vs_target(
    df,
    target_col="target",
    output_dir="reports/figures"
)

In [13]:
shift_df = compute_split_shift(
    df,
    split_col="split",
    save_path="reports/tables/feature_split_shift.csv"
)

shift_df

,feature,train_mean,val_mean,test_mean,train_std,val_std,test_std
0,CFL.img,0.011040,-0.016845,-0.025168,0.963988,1.052868,1.076865
1,FSP_Index.img,-0.006438,-0.026411,0.039218,1.048653,1.149360,0.670993
2,Ignition_Prob.img,-0.008964,-0.034541,0.053093,1.046362,1.149791,0.680968
3,Struct_Exp_Index.img,-0.006438,-0.026411,0.039218,1.048653,1.149360,0.670993
4,Fuel_Models.img,137.506073,134.568839,138.651934,28.525573,27.041474,32.042842
5,row_norm,0.463453,0.358203,0.269890,0.260794,0.103383,0.194890
6,col_norm,0.418512,0.288919,0.296451,0.197005,0.150939,0.126662
7,target,0.028108,0.030748,0.011571,0.036420,0.032690,0.015482
8,row_index,880.096542,680.227442,512.521317,495.247686,196.323702,370.095713
9,col_index,790.150733,545.479690,559.700358,371.946129,284.973451,239.136936
